# Support Vector Classifiers

Here we try many support vector classifiers, with different kernels. 

In [ ]:
import pandas as pd

learn_data = pd.read_csv("../data/preprocess_train_v3.csv", header = None)
learn_data.columns = ['Age', 'ALB', 'AR', 'IBRatio', 'AST_ALT_Ratio', 'LogIB', 'LogAlkphos', 'LogSgpt', 'LogSgot', 'Female', 'Target']
learn_data["Female"] = learn_data["Female"].astype("category")
learn_data["Target"] = learn_data["Target"].astype("category")
learn_data.head()

,Age,ALB,AR,IBRatio,AST_ALT_Ratio,LogIB,LogAlkphos,LogSgpt,LogSgot,Female,Target
0,48,2.4,0.52,0.488889,5.692308,7.884574e-01,5.641907,2.564949,4.304065,0,0
1,39,4.3,1.38,0.526316,1.476190,-1.110223e-16,5.192957,3.737670,4.127134,0,0
2,23,3.1,1.00,0.700000,1.951220,-3.566749e-01,5.356586,3.713572,4.382027,0,0
3,42,3.2,1.06,0.714286,2.314286,-6.931472e-01,5.023881,3.555348,4.394449,1,0
4,54,3.4,0.80,0.495575,1.233333,2.415914e+00,6.324359,3.401197,3.610918,1,0


In [2]:
from sklearn.model_selection import train_test_split

X = learn_data.drop(columns = ["Target"])
Xnum = X.drop(columns = ["Female"])
y = learn_data["Target"]

X_train, X_val, Xnum_train, Xnum_val, y_train, y_val = train_test_split(X, Xnum, y, test_size = 0.33, random_state = 42)

## Metrics

In [3]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
import numpy as np

def compute_metrics (y_real, y_pred) -> list[float]:
    F1_macro = f1_score(y_real, y_pred, average = "macro")
    recall = recall_score(y_real, y_pred, average = "macro")
    prec = precision_score(y_real, y_pred, average = "macro")
    acc = accuracy_score(y_real, y_pred)
    return [F1_macro, recall, prec, acc]

def confusion (y_real, y_pred) -> None:
    TP = sum(np.logical_and(y_real == y_pred, y_real == 1))
    TN = sum(np.logical_and(y_real == y_pred, y_real == 0))
    FP = sum(np.logical_and(y_real != y_pred, y_real == 0))
    FN = sum(np.logical_and(y_real != y_pred, y_real == 1))
    print("\t\tPredicted")
    print("\t\t+1\t0")
    print(f"Real\t+1\t{TP}\t{FN}")
    print(f"\t0\t{FP}\t{TN}")
    print(f"Accuracy: {((TP + TN) / y_real.shape[0] * 100):.2f}%".format())

metrics_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

## Linear kernel (or no kernel)

We have seen with other linear classifiers that the performance is not very good because of two reasons:
- Excessive resampling: we might be resampling too much, and this may affect our predictive power by creating samples that do not correspond to the real data.

In [4]:
from sklearn.svm import LinearSVC

linear_model = LinearSVC(class_weight = "balanced")
linear_model.fit(Xnum_train, y_train)

confusion(np.array(y_train), pd.Series(linear_model.predict(Xnum_train)))

		Predicted
		+1	0
Real	+1	65	12
	0	97	128
Accuracy: 63.91%


In [5]:
confusion(np.array(y_val), pd.Series(linear_model.predict(Xnum_val)))

		Predicted
		+1	0
Real	+1	44	8
	0	42	55
Accuracy: 66.44%


In [6]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

linear_model = LinearSVC(class_weight = "balanced")
linsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", linear_model)])

n = 100
Cs = np.logspace(start = -1, stop = 2, num = n)

linsvc_search = GridSearchCV(estimator = linsvc_pipeline,
                             param_grid = {'svc__C' : Cs},
                             scoring = 'f1_macro',
                             cv = 5)
linsvc_search.fit(Xnum, y)
linsvc_search.best_params_

{'svc__C': 0.09999999999999999}

In [7]:
linsvc_search.best_score_

0.6454264748724517

In [8]:
from sklearn.model_selection import cross_validate

linsvc_C = linsvc_search.best_params_['svc__C']
linsvc_best = LinearSVC(C = linsvc_C, class_weight = "balanced")
linsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", linsvc_best)])

cross_val_results = pd.DataFrame(cross_validate(linsvc_pipeline, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["Linear SVC", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Linear SVC,0.645426,0.70525,0.66823,0.658584


## Gaussian kernel

In [9]:
from sklearn.svm import SVC

rbf_scale_model = SVC(kernel = "rbf", gamma = "scale", class_weight = "balanced")
rbf_scale_model.fit(Xnum_train, y_train)

confusion(np.array(y_train), pd.Series(rbf_scale_model.predict(Xnum_train)))

		Predicted
		+1	0
Real	+1	41	36
	0	85	140
Accuracy: 59.93%


In [10]:
confusion(np.array(y_val), pd.Series(rbf_scale_model.predict(Xnum_val)))

		Predicted
		+1	0
Real	+1	30	22
	0	35	62
Accuracy: 61.74%


In [11]:
rbfsvc = SVC(kernel = "rbf", class_weight = "balanced")
rbfsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", rbfsvc)])

n = 50
m = 50
Cs = np.logspace(start = -1, stop = 2, num = n)
gammas = np.logspace(start = -1, stop = 1, num = m) / X.shape[0]

rbfsvc_search = GridSearchCV(estimator = rbfsvc_pipeline,
                             param_grid = {'svc__C' : Cs,
                                           'svc__gamma' : gammas},
                             scoring = 'f1_macro',
                             cv = 5)
rbfsvc_search.fit(Xnum, y)
rbfsvc_search.best_params_

{'svc__C': 100.0, 'svc__gamma': 0.022172949002217297}

In [12]:
rbfsvc_search.best_score_

0.6289796130888243

In [13]:
rbfsvc_C = rbfsvc_search.best_params_['svc__C']
rbfsvc_gamma = rbfsvc_search.best_params_['svc__gamma']
rbfsvc_best = SVC(kernel = "rbf", C = rbfsvc_C, gamma = rbfsvc_gamma, class_weight = "balanced")
rbfsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", rbfsvc_best)])

cross_val_results = pd.DataFrame(cross_validate(rbfsvc_pipeline, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["Gaussian SVC", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Linear SVC,0.645426,0.70525,0.66823,0.658584
Gaussian SVC,0.62898,0.69275,0.659057,0.64083


## Polynomial kernel

In [14]:
poly_model = SVC(kernel = "poly", degree = 2, gamma = "scale", class_weight = "balanced")
poly_model.fit(Xnum_train, y_train)

confusion(np.array(y_train), pd.Series(poly_model.predict(Xnum_train)))

		Predicted
		+1	0
Real	+1	58	19
	0	128	97
Accuracy: 51.32%


In [15]:
confusion(np.array(y_val), pd.Series(poly_model.predict(Xnum_val)))

		Predicted
		+1	0
Real	+1	37	15
	0	51	46
Accuracy: 55.70%


In [16]:
polysvc = SVC(kernel = "poly", class_weight = "balanced")
polysvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", polysvc)])

n = 25
m = 25
Cs = np.logspace(start = -1, stop = 2, num = n)
gammas = np.logspace(start = -2, stop = 2, num = m) / X.shape[0]
degrees = [2, 3]

polysvc_search = GridSearchCV(estimator = polysvc_pipeline,
                              param_grid = {'svc__C' : Cs,
                                            'svc__gamma' : gammas,
                                            'svc__degree' : degrees},
                              scoring = 'f1_macro',
                              cv = 5)
polysvc_search.fit(Xnum, y)
polysvc_search.best_params_

{'svc__C': 1.0, 'svc__degree': 3, 'svc__gamma': 0.22172949002217296}

In [17]:
polysvc_search.best_score_

0.6438724887264813

In [18]:
polysvc_C = polysvc_search.best_params_['svc__C']
polysvc_gamma = polysvc_search.best_params_['svc__gamma']
polysvc_degree = polysvc_search.best_params_['svc__degree']
polysvc_best = SVC(kernel = "poly",
                   C = polysvc_C,
                   gamma = polysvc_gamma,
                   degree = polysvc_degree,
                   class_weight = "balanced")
polysvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", polysvc_best)])

cross_val_results = pd.DataFrame(cross_validate(polysvc_pipeline, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["Polynomial SVC", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Linear SVC,0.645426,0.70525,0.66823,0.658584
Polynomial SVC,0.643872,0.69499,0.66014,0.660733
Gaussian SVC,0.62898,0.69275,0.659057,0.64083


## Sigmoid

In [19]:
sig_model = SVC(kernel = "sigmoid", gamma = "scale", class_weight = "balanced")
sig_model.fit(Xnum_train, y_train)

confusion(np.array(y_train), pd.Series(sig_model.predict(Xnum_train)))

		Predicted
		+1	0
Real	+1	44	33
	0	105	120
Accuracy: 54.30%


In [20]:
confusion(np.array(y_val), pd.Series(sig_model.predict(Xnum_val)))

		Predicted
		+1	0
Real	+1	28	24
	0	42	55
Accuracy: 55.70%


In [21]:
sigsvc = SVC(kernel = "sigmoid", class_weight = "balanced")
sigsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", sigsvc)])

n = 50
m = 50
Cs = np.logspace(start = -1, stop = 2, num = n)
gammas = np.logspace(start = -2, stop = 2, num = m) / X.shape[0]

sigsvc_search = GridSearchCV(estimator = sigsvc_pipeline,
                             param_grid = {'svc__C' : Cs,
                                            'svc__gamma' : gammas},
                             scoring = 'f1_macro',
                             cv = 5)
sigsvc_search.fit(Xnum, y)
sigsvc_search.best_params_

{'svc__C': 75.43120063354614, 'svc__gamma': 0.015957553725080963}

In [25]:
sigsvc_search.best_score_

0.6453528289148487

In [26]:
sigsvc_C = sigsvc_search.best_params_['svc__C']
sigsvc_gamma = sigsvc_search.best_params_['svc__gamma']
sigsvc_best = SVC(kernel = "sigmoid",
                  C = sigsvc_C,
                  gamma = sigsvc_gamma,
                  class_weight = "balanced")
sigsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", sigsvc_best)])

cross_val_results = pd.DataFrame(cross_validate(sigsvc_pipeline, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["Sigmoid SVC", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Linear SVC,0.645426,0.70525,0.66823,0.658584
Sigmoid SVC,0.645353,0.699913,0.663325,0.66083
Polynomial SVC,0.643872,0.69499,0.66014,0.660733
Gaussian SVC,0.62898,0.69275,0.659057,0.64083


## Trying our best models on test dataset


In [ ]:
test_data = pd.read_csv("../data/preprocess_test_v3.csv", header = None)
test_data.columns = ['Age', 'ALB', 'AR', 'IBRatio', 'AST_ALT_Ratio', 'LogIB', 'LogAlkphos', 'LogSgpt', 'LogSgot', 'Female']
test_data["Female"] = learn_data["Female"].astype("category")
test_data.head()

,Age,ALB,AR,IBRatio,AST_ALT_Ratio,LogIB,LogAlkphos,LogSgpt,LogSgot,Female
0,11,4.2,1.40,0.857143,1.115385,-0.510826,6.383507,3.258097,3.367296,0
1,62,4.0,0.80,0.500000,2.246377,-0.105361,5.411646,4.234107,5.043425,0
2,60,4.2,1.10,0.714286,0.437500,-0.693147,5.159055,3.465736,2.639057,0
3,60,3.2,0.78,0.508772,2.063107,1.064711,5.365976,6.021023,6.745236,1
4,48,2.7,0.90,0.777778,2.250000,-0.356675,5.164786,3.178054,3.988984,1


In [29]:
test_data_num = test_data.drop(columns = ["Female"])

### Linear kernel

In [31]:
linsvc_pipeline.fit(Xnum, y)

labels_lin = pd.DataFrame(columns = ['ID', 'Label'])
labels_lin['Label'] = pd.DataFrame(linsvc_pipeline.predict(test_data_num))
labels_lin['ID'] = labels_lin.index + 1
labels_lin

,ID,Label
0,1,1
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,0
112,113,0
113,114,1
114,115,1


### Gaussian kernel (no categorical)

In [33]:
rbfsvc_pipeline.fit(Xnum, y)

labels_rbf = pd.DataFrame(columns = ['ID', 'Label'])
labels_rbf['Label'] = pd.DataFrame(rbfsvc_pipeline.predict(test_data_num))
labels_rbf['ID'] = labels_rbf.index + 1
labels_rbf

,ID,Label
0,1,0
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,1
112,113,0
113,114,1
114,115,1


In [ ]:
labels_rbf.to_csv('../data/new_predictions/rbfsvc_best.csv', index = False)

### Polynomial kernel

In [36]:
polysvc_pipeline.fit(Xnum, y)

labels_poly = pd.DataFrame(columns = ['ID', 'Label'])
labels_poly['Label'] = pd.DataFrame(polysvc_pipeline.predict(test_data_num))
labels_poly['ID'] = labels_rbf.index + 1
labels_poly

,ID,Label
0,1,1
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,1
112,113,1
113,114,1
114,115,1


In [ ]:
labels_poly.to_csv('../data/new_predictions/polysvc_best.csv', index = False)

### Sigmoid kernel

In [39]:
sigsvc_pipeline.fit(Xnum, y)

labels_sig = pd.DataFrame(columns = ['ID', 'Label'])
labels_sig['Label'] = pd.DataFrame(sigsvc_pipeline.predict(test_data_num))
labels_sig['ID'] = labels_sig.index + 1
labels_sig

,ID,Label
0,1,1
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,1
112,113,1
113,114,1
114,115,1


In [ ]:
labels_sig.to_csv('../data/new_predictions/sigsvc_best.csv', index = False)

### Discrepancies between models

None at all between gaussian and sigmoid kernels, but a lot of discrepancy with the gaussian

In [43]:
(labels_rbf == labels_sig).value_counts()

ID    Label
True  True     90
      False    26
Name: count, dtype: int64

In [44]:
(labels_rbf == labels_lin).value_counts()

ID    Label
True  True     97
      False    19
Name: count, dtype: int64

In [46]:
(labels_sig == labels_poly).value_counts()

ID    Label
True  True     90
      False    26
Name: count, dtype: int64